In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0.1)
prompt = PromptTemplate.from_template("What's the weather in {city}")
chain = prompt | llm
chain.invoke({"city":"rome"})

"""
response:
Sorry, CANNOT provide real-time info...
"""

In [ ]:
import json
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

def get_weather(lon, lat):
    print("calling an API")

function = {
    "name": "get_weather",
    "description": "function that takes longitude and latitude to find a weather of the place.",
    "parameters": {
        "type": object,
        "properties": {
            "lon": {"type": "float", "description": "The longitude coordinate"},
            "lat": {"type": "float", "description": "The latitude coordinate"},
        },
    },
    "required": ["lon", "lat"],
}

llm = ChatOpenAI(temperature=0.1).bind(
    function_call={ 
        "name": "get_weather" # 함수를 사용하도록 강제
    }, # model이 스스로 선택하게 하려면 function_call="auto"
    functions=[function],
)
prompt = PromptTemplate.from_template("What's the weather in {city}")
chain = prompt | llm
response = chain.invoke({"city":"rome"})

"""
response:
AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments':'{"lon":"12.4", "lat":"41.9"}'}})
"""

response = response.additional_kwargs["function_call"]["arguments"]
"""
response:
'{"lon":"12.4", "lat":"41.9"}'
"""

r = json.loads(response)
"""
response:
{'lon':'12.4', 'lat':'41.9'}
"""
get_weather(r['lon'], r['lat'])

In [ ]:
import json
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

schema = {
    "name":"create_quiz",
    "description":"function that takes a list of questions and answers and returns a quiz",
    "parameters":{
        "type":"object",
        "properties":{"questions":{
            "type":"array",
            "items":{
                "type":"object",
                "properties":{
                    "question":{"type":"string"},
                    "answers":{
                        "type":"array",
                        "items":{
                            "type":"object",
                            "properties":{
                                "answer":{"type":"string"},
                                "correct":{"type":"boolean"},
                            },
                            "required":["answer", "correct"],
                        },
                    },
                },
                "required":["question", "answers"],
            },
        }},
        "required":["questions"],
    },
}

llm = ChatOpenAI(temperature=0.1).bind(
    functions=[schema],
    function_call={"name":"create_quiz"},
)

prompt = PromptTemplate.from_template("Make a quiz about {city}")

chain = prompt | llm
response = chain.invoke({"city":"Rome"})
response = response.additional_kwargs["function_call"]["arguments"]

for question in json.loads(response)["questions"]:
    print(question)